# Running a DAS document, and watching the plan fill in

One document, start to finish: JSON on disk → a validated `Spec` → a plan → one nnsight
session → results sitting on the plan → files.

The document is [`documents/v2/das.json`](../documents/v2/das.json), written in
causalab-mini's steps-first format. A fit trains a rank-8 rotation at layer 0 of a tiny
CPU Llama — its body is the experiment, run once per minibatch — then the same two
declared interventions score the trained rotation, and the rotation is saved.

Everything here runs on CPU in about ten seconds.

## 1. The document

In [1]:
import json, pathlib, pickle, torch

REPO = pathlib.Path.cwd().parent
raw = json.loads((REPO / 'documents' / 'v2' / 'das.json').read_text())

print('root keys :', list(raw))
print('steps     :', list(raw['steps']))
print('fit body  :', list(raw['steps']['fit']['steps']))
print('saves     :', raw['steps']['saves'])

root keys : ['header', 'model', 'data', 'sites', 'featurizers', 'interventions', 'steps']
steps     : ['fit', 'counterfactual', 'patched', 'iia', 'ce', 'saves']
fit body  : ['counterfactual', 'patched', 'iia', 'ce']
saves     : {'fit.iia': 'held_out_iia.json', 'iia': 'iia.json', 'ce': 'ce.json', 'fit.rot': 'rot.safetensors'}


`Spec` is a pydantic model, so parsing *is* validating. An unknown key anywhere, a site
that isn't declared, a reference to a step that has not run yet — each is refused here,
with the path to it, before anything is loaded.

In [2]:
from causalab_mini.plan.spec import Spec

spec = Spec.model_validate(raw)
spec.steps['fit'].early_stop

EarlyStop(metric='iia', mode='max', patience=3)

In [3]:
# what a refusal looks like: an unknown key, and a trained rotation named before its fit
from pydantic import ValidationError

broken = json.loads(json.dumps(raw))
broken['interventions']['cf_read']['reads']['v_cf']['shuffle'] = {'seed': 1}
try:
    Spec.model_validate(broken)
except ValidationError as refusal:
    print(refusal)

early = json.loads(json.dumps(raw))
early['steps'] = {'peek': {'kind': 'forward', 'data': 'train', 'field': 'input', 'interventions': 'cf_read'}, **early['steps']}
try:
    Spec.model_validate(early)
except ValidationError as refusal:
    print(refusal)

1 validation error for Spec
interventions.cf_read.reads.v_cf.shuffle
  Extra inputs are not permitted [type=extra_forbidden, input_value={'seed': 1}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
1 validation error for Spec
  Value error, step 'peek': read 'v_cf': featurizer 'fit.rot' is used before step 'fit' trains it, so it would run on the untrained parameter. Move it after the fit — or, for a deliberate untrained baseline, declare a second featurizer that no fit names [type=value_error, input_value={'header': {'description'...t': 'rot.safetensors'}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## 2. The engine loads the model

An engine *is* a runtime: it loads the model and it knows how to reach inside it. Here
that's nnterp's `StandardizedTransformer` under nnsight.

In [4]:
from causalab_mini.engine import NNterpEngine

engine = NNterpEngine.load(spec.model, device_map='cpu')
print(type(engine.model).__name__, '|', engine.num_layers, 'layers |', engine.width(engine.locate('block_output', 0)), 'wide')

StandardizedTransformer | 2 layers | 16 wide


## 3. Compiling: document → plan

This is the only place the model is consulted on the client. It resolves what the block
may not decide for itself — the addresses, the widths, the tokens, which rows. The
plan's steps are the document's, one for one.

In [5]:
from causalab_mini import plan as plan_module

plan = plan_module.build_request(raw, REPO / 'documents' / 'data', engine)
list(plan.steps)

['featurizers', 'fit', 'fit.weights', 'counterfactual', 'patched', 'iia', 'ce']

### The tree

`featurizers` is the step the compiler adds — declaring a parameter set is what builds
it. `fit` is the fit, and `fit.weights` the rotation it trained, published there because
a save names `fit.rot`. Then the scoring: two forwards and two metrics, each its own
step. `explain` prints the whole tree:

In [6]:
from causalab_mini.plan.explain import explain

print(explain(plan))

<root>: Plan
  featurizers: Featurizers
      rot: subspace k=8 d=16 cayley seed=0 trained=True
  fit: Fit  10 epochs x 1 update  lr=0.001 objective=((1.0, 'ce'),) params=('rot',)  early_stop='iia' patience=3
      epochs[0][0]: Plan
        counterfactual: forward on 'train'  (2 rows x 11 tokens)
            read  'counterfactual.v_cf' at block_output[0] pos={index:-1} via 'rot'
        patched: forward on 'train'  (2 rows x 11 tokens)
            write 'patched.patch' at block_output[0] pos={index:-1} swap(counterfactual.v_cf) via 'rot'
            read  'patched.logits' at lm_head pos={index:-1} via 'identity'
        iia: metric logit_diff(patched.logits)
        ce: metric cross_entropy(patched.logits)
      evaluation: Plan
        counterfactual: forward on 'train'  (2 rows x 9 tokens)
            read  'counterfactual.v_cf' at block_output[0] pos={index:-1} via 'rot'
        patched: forward on 'train'  (2 rows x 9 tokens)
            write 'patched.patch' at block_output[0] po

Three things to notice.

**`d=16` was never authored.** The document says `k: 8`; the width came from asking the
engine how wide `block_output` is on this model.

**A fit's body is its own scope.** Every update is a plan of the body's steps, named as
the body names them — `counterfactual`, `patched`, `iia`, `ce` — and run in a scope of
its own, so the body can reuse the root's step names, and what it reads stays attached
to its graph until the update's optimizer step.

**Positions are specs.** The document says `pos: -1` and so does the plan. The run
resolves it against the model's own tokenizer, per row, inside the session.

### A plan is data

Before it runs, the whole tree is strings, integers and tuples — no tensors, no model,
no tokenizer. That is what makes it shippable to a remote server.

In [7]:
print('pickles with plain pickle:', pickle.loads(pickle.dumps(plan)) == plan)
print('results before the run    :', {name: step.results for name, step in plan.steps.items()})

pickles with plain pickle: True
results before the run    : {'featurizers': {}, 'fit': {}, 'fit.weights': {}, 'counterfactual': {}, 'patched': {}, 'iia': {}, 'ce': {}}


## 4. Running it

One `model.session(...)` for the whole request — the fit included. `remote=True` would
be the only change needed to run this on NDIF.

In [8]:
from causalab_mini.plan import Fit

executed = engine.execute(plan)

for name, step in executed.steps.items():
    print(f'{name:14s} {sorted(step.results)}')
print(f"{'fit evaluation':14s} {sorted(executed.step('fit', Fit).evaluation.all_results())}")

featurizers    []
fit            ['train/eval', 'train/loss']
fit.weights    ['rot']
counterfactual []
patched        []
iia            ['iia']
ce             ['ce']
fit evaluation ['ce', 'iia']


### The results are on the nodes that produced them

The fit's body and the scoring after it both have a metric step called `iia`. One is on
the fit's evaluation — a document saves it as `fit.iia` — and the other is the root's: two
places, two references.

In [9]:
fit = executed.step('fit', Fit)

print('scored  (trained-on rows) iia =', executed.result('iia').tolist())
print('held-out (unseen rows)    iia =', fit.evaluation.result('iia').tolist())
print()
print('loss per update :', fit.results['train/loss'].tolist())
print('eval per epoch  :', fit.results['train/eval'].squeeze(-1).tolist())

scored  (trained-on rows) iia = [0.011959902942180634, -0.002783656120300293]
held-out (unseen rows)    iia = [0.22377648949623108, -0.22890233993530273]

loss per update : [10.469064712524414, 10.469003677368164, 10.468942642211914, 10.46888256072998]
eval per epoch  : [-0.002514287829399109, -0.0025305449962615967, -0.002546735107898712, -0.0025629252195358276]


The fit stopped after 4 epochs of a 10-epoch budget: it early-stops on `iia`, `iia` fell
on every evaluation, and patience is 3. You can read every update where it happened:

In [10]:
for e, epoch in enumerate(fit.epochs[:4]):
    for update in epoch:
        print(f'epoch {e}:', {k: [round(x, 4) for x in v.tolist()] for k, v in update.all_results().items()})

epoch 0: {'iia': [-0.0033, 0.0115], 'ce': [10.4726, 10.4655]}
epoch 1: {'iia': [0.0116, -0.0031], 'ce': [10.4655, 10.4725]}
epoch 2: {'iia': [-0.003, 0.0117], 'ce': [10.4724, 10.4654]}
epoch 3: {'iia': [-0.0029, 0.0118], 'ce': [10.4724, 10.4654]}


### The rotation

In [11]:
from causalab_mini.ops import featurizer

weight = executed.result('rot')
basis = featurizer.cayley(weight)
print('weight       :', tuple(weight.shape))
print('QᵀQ == I     :', torch.allclose(basis.T @ basis, torch.eye(8), atol=1e-5))
print('moved off its start:', not torch.equal(weight, featurizer.start_weight(16, 8, 0)))

weight       : (16, 8)
QᵀQ == I     : True
moved off its start: True


## 5. What leaves the run

`Plan.write` walks the tree. Each reference `steps.saves` names is written from the node
that produced it, so the held-out score lands beside the scored one.

In [12]:
import tempfile

out = pathlib.Path(tempfile.mkdtemp())
for path in executed.write(out):
    print(path.relative_to(out))

document.json
run.json
held_out_iia.json
rot.safetensors
iia.json
ce.json


In [13]:
rows = json.loads((out / 'held_out_iia.json').read_text())
rows

[{'example_id': '0', 'metric': 'iia', 'value': 0.22377648949623108, 'eligible': True, 'positions': None, 'reason': '', 'tokens': '', 'unit': 'logit', 'estimand_version': 'logit_diff/v1', 'produced_by': 'dd4aa6261790348e7a56642f752ddc1f471e708fe939351dbd33cdd0773ff182'}, {'example_id': '1', 'metric': 'iia', 'value': -0.22890233993530273, 'eligible': True, 'positions': None, 'reason': '', 'tokens': '', 'unit': 'logit', 'estimand_version': 'logit_diff/v1', 'produced_by': 'dd4aa6261790348e7a56642f752ddc1f471e708fe939351dbd33cdd0773ff182'}]

## 6. The same experiment, in the protocol's format

[`documents/das_cpu_reduction.json`](../documents/das_cpu_reduction.json) is this run
written in causalab's own JSON. Different front end, same compiler underneath — so the
same rotation, to the bit.

In [14]:
protocol_raw = json.loads((REPO / 'documents' / 'das_cpu_reduction.json').read_text())
other = engine.execute(plan_module.build_request(protocol_raw, REPO / 'documents' / 'data', engine))

print('same rotation:', torch.equal(executed.result('rot'), other.result('rot')))
print('same scores  :', torch.equal(executed.result('iia'), other.result('iia')))
print()
print('but the protocol version cannot save the held-out number:')
print('  its saves ->', [s.file_path for one in other.steps.values() for s in one.saves])

same rotation: True
same scores  : True

but the protocol version cannot save the held-out number:
  its saves -> ['iia.json', 'ce.json', 'rot.safetensors']
